# AD existentes · XTTS + Spanish-F5 + gráficas

Este notebook utiliza las AD de `datos/Prueba 2 VDMS (XTTS)/1. Input`.
**No vuelve a diarizar, analizar imágenes ni llamar a APIs. No necesita claves.**
Lee `main_summary` y `complementary_emotion`, verifica los videos y referencias,
permite revisar el texto, genera ambos motores y calcula los gráficos comparativos.

**10 videos × 3 locutores × 2 motores = 60 audios nuevos y 30 pares.**
Los audios XTTS históricos no se mezclan con la nueva comparación. Se comparan WAV raw,
antes de ajustes de velocidad o ducking. La mezcla de video es opcional.

Sube la carpeta **V01 Original** a `Mi unidad/EmotiveAD/V01 Original` en Drive:
incluye `emotivead`, `config`, los scripts `.py`, los JSON de entrada,
`datos/Locutores` y `datos/videos prueba mp4`. No subas `.engines`, entornos virtuales
ni credenciales. Activa GPU en Colab y ejecuta las celdas en orden.

In [ ]:
# @title 1. Drive, rutas e instalación del núcleo
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, subprocess, shutil
# Ejecuta un comando y muestra su salida en vivo (Colab oculta la salida de los subprocesos).
import subprocess as _sp, collections as _co
def run(cmd, env=None):
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd[:4]), '...' if len(cmd) > 4 else '')
    proc = _sp.Popen(cmd, stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True, bufsize=1, env=env)
    tail = _co.deque(maxlen=80)
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
    if proc.wait():
        raise RuntimeError('El comando falló (código %d). Últimas líneas:\n%s' % (proc.returncode, ''.join(tail)))
V01 = Path('/content/drive/MyDrive/EmotiveAD/V01 Original') # @param {type:"string"}
DATA = V01/'datos'
INPUT = DATA/'Prueba 2 VDMS (XTTS)'/'1. Input'
VIDEOS = DATA/'videos prueba mp4'
REFS = DATA/'Locutores'
OUT = V01/'resultados_AD_existentes'
for path in [V01/'prepare_existing_ad.py', V01/'run_pipeline.py', INPUT, VIDEOS, REFS]:
    assert path.exists(), f'No se encuentra {path}. Sube la carpeta completa o corrige V01.'
run(['apt-get','-qq','update'])
run(['apt-get','-qq','install','-y','ffmpeg'])
run([sys.executable,'-m','pip','install','uv','ipywidgets'])
UV = shutil.which('uv'); PY = '/content/emotive-core/bin/python'
if not Path(PY).exists(): run([UV,'venv','--python','3.11','/content/emotive-core'])
run([UV,'pip','install','--python',PY,'-r',str(V01/'requirements.txt')])
os.environ['PYTHONPATH'] = str(V01)
# Colab define un backend inline de matplotlib que los entornos de XTTS/F5 no tienen.
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONUNBUFFERED'] = '1'  # muestra el avance en vivo
print('Entradas:',INPUT,'\nResultados:',OUT)

In [ ]:
# @title 2. Validar las AD existentes y las voces de referencia
NARRATORS = ['Woman01','Man01','Man02']
import re, unicodedata
def _k(s):
    s = unicodedata.normalize('NFKD', s)
    return re.sub(r'[^a-z0-9]', '', ''.join(c for c in s if not unicodedata.combining(c)).lower())

# --- Revisión previa de Drive: dice exactamente qué falta ---
problemas = []
carpetas = sorted(p for p in INPUT.iterdir() if p.is_dir())
print('Carpetas en 1. Input:', len(carpetas))
for c in carpetas:
    if c.name.startswith('.'):
        problemas.append(f'Carpeta oculta en 1. Input (bórrala): {c.name}'); continue
    for f in ['video_summary.json', 'prosody_analysis.json']:
        if not (c/f).exists(): problemas.append(f'Falta {c.name}/{f}')
    if not (VIDEOS/(c.name+'.mp4')).exists(): problemas.append(f'Falta el video {c.name}.mp4 en "videos prueba mp4"')
for n in NARRATORS:
    wavs = list((REFS/n).glob('*.wav')) + list((REFS/n).glob('*.WAV'))
    print(f'{n}: {len(wavs)} WAV')
    if len(wavs) != 16: problemas.append(f'{n} tiene {len(wavs)} WAV (se esperan 16)')
    claves = [_k(re.sub(r'^\d+\s*', '', w.stem)) for w in wavs]
    for cl in set(claves):
        if claves.count(cl) > 1: problemas.append(f'{n}: referencia duplicada "{cl}" (¿archivo "(1)" de Drive?)')
    for w in wavs:
        if w.suffix == '.WAV': problemas.append(f'{n}/{w.name}: extensión en mayúsculas, renómbrala a .wav')
        if w.stat().st_size < 1000: problemas.append(f'{n}/{w.name}: archivo vacío o incompleto (¿Drive sigue subiendo?)')
if problemas:
    print('\nPROBLEMAS ENCONTRADOS EN DRIVE:'); [print(' -', x) for x in problemas]

r = subprocess.run([PY,str(V01/'prepare_existing_ad.py'),'--input',str(INPUT),'--videos',str(VIDEOS),
    '--references',str(REFS),'--output',str(OUT),'--narrators',*NARRATORS], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('ERROR REAL DEL SCRIPT:\n', r.stderr[-3000:])
    raise SystemExit('Corrige lo indicado arriba y vuelve a ejecutar esta celda.')
audit = json.loads((OUT/'text_preparation_audit.json').read_text())
for row in audit:
    print(row['video_id'], '|',row['emotion'],'|',round(row['duration_s'],2),'s')
    print(row['narration'])
    if row['changed']: print('  Se retiraron marcas o notas de análisis; original conservado en el registro.')
print('AD:',len(audit),'| Audios esperados:',len(audit)*len(NARRATORS)*2)

In [ ]:
# @title 3. Revisar las AD (recupera tu revisión si ya existe)
import ipywidgets as widgets
from IPython.display import display, Audio, Video, Image
REVIEWED = OUT/'projects_reviewed.json'
draft = REVIEWED if REVIEWED.exists() else OUT/'projects_draft.json'
editor = widgets.Textarea(value=draft.read_text(encoding='utf-8'),
    layout=widgets.Layout(width='100%',height='450px'))
display(editor)
print('Puedes corregir text y emotion. Ambos motores utilizarán exactamente esta misma revisión.')

In [ ]:
# @title 4. Guardar y validar la revisión antes de instalar los motores
PROJECTS = json.loads(editor.value)
expected_ids = {p['video_id'] for p in json.loads((OUT/'projects_draft.json').read_text())}
assert len(PROJECTS)==len(expected_ids) and {p['video_id'] for p in PROJECTS}==expected_ids, 'Conserva los videos de entrada y sus identificadores.'
REVIEWED.write_text(json.dumps(PROJECTS,ensure_ascii=False,indent=2),encoding='utf-8')
check = """
import sys
from pathlib import Path
from emotivead.contracts import read_json, validate_segments, reference_for
from emotivead.media import probe
projects=read_json(sys.argv[1]); original={p['video_id']:p for p in read_json(sys.argv[3])}
for p in projects:
    assert p['video']==original[p['video_id']]['video'], 'Conserva la ruta del video.'
    p['segments']=validate_segments(p['segments'],probe(p['video'])['duration'])
    for s in p['segments']:
        for narrator in sys.argv[4:]: reference_for(Path(sys.argv[2]),narrator,s['emotion'])
print('Textos, emociones, tiempos, videos y referencias validados.')
"""
run([PY,'-c',check,str(REVIEWED),str(REFS),str(OUT/'projects_draft.json'),*NARRATORS])
EXPECTED = sum(len(p['segments']) for p in PROJECTS)*len(NARRATORS)*2
print('Se generarán',EXPECTED,'audios raw.')

In [ ]:
# @title 5. Instalar XTTS y Spanish-F5 en entornos independientes
# Lee la licencia: https://huggingface.co/coqui/XTTS-v2/blob/main/LICENSE.txt
ACCEPT_XTTS_LICENSE = False # @param {type:"boolean"}
if not ACCEPT_XTTS_LICENSE: raise ValueError('Lee la licencia CPML de XTTS y marca la casilla si aceptas sus condiciones.')
os.environ['COQUI_TOS_AGREED']='1'
print('Espacio libre en /content:', round(shutil.disk_usage('/content').free/1e9,1), 'GB (se necesitan ~20 GB)')
# Un motor a la vez: si uno falla, el mensaje dice cuál y el otro queda instalado.
for engine in ['xtts','spanish_f5']:
    print(f'\n===== Instalando {engine} =====')
    run([PY, V01/'setup_envs.py', '--engines', engine, '--target', '/content/emotive-engines'])
ENV = '/content/emotive-engines/env.json'
envs = json.loads(Path(ENV).read_text())
for engine in ['xtts','spanish_f5']:
    run([envs['EMOTIVE_'+engine.upper()+'_PYTHON'],'-c',
        'import torch; assert torch.cuda.is_available(), "Activa GPU en Colab"; print(torch.cuda.get_device_name())'])

In [ ]:
# @title 6. Generar las AD con los dos motores
GENERATE_MIXED_VIDEO = False # @param {type:"boolean"}
cmd=[PY,str(V01/'run_pipeline.py'),'--root',str(DATA),'--output',str(OUT),
    '--projects',str(REVIEWED),'--option','3','--env',ENV,'--narrators',*NARRATORS]
if GENERATE_MIXED_VIDEO: cmd.append('--mix')
run(cmd)
records=json.loads((OUT/'audio_manifest.json').read_text())
assert len(records)==EXPECTED, 'Faltan audios. Revisa failures.json y los logs antes de evaluar.'
for engine in ['xtts','spanish_f5']:
    print(engine,sum(r['engine']==engine for r in records),'audios')
print('Si Colab se interrumpe, monta Drive, recupera la revisión, instala entornos y repite esta celda. Se comprueban hashes antes de reutilizar WAV.')

In [ ]:
# @title 7. Escuchar un video en los dos motores y los tres locutores
VIDEO_ID = PROJECTS[0]['video_id'] # Cambia por el identificador que quieras comparar.
for record in records:
    if record['video_id']==VIDEO_ID:
        print(record['engine'],record['narrator'],record['emotion'])
        display(Audio(filename=record['path']))
if GENERATE_MIXED_VIDEO:
    report=json.loads((OUT/VIDEO_ID/'mix_report.json').read_text())
    for item in report['completed']:
        print(item['engine'],item['narrator']); display(Video(filename=item['video'],embed=True,width=640))
    for item in report['needs_edit']: print('La voz no cabe en el clip:',item)

In [ ]:
# @title 8. Calcular medidas acústicas y gráficas comparativas
REPORT = OUT/'graficas_comparacion_raw'
run([PY,str(V01/'evaluate.py'),'--root',str(DATA),'--manifest',str(OUT/'audio_manifest.json'),
    '--stage','raw','--output',str(REPORT)])
errors=json.loads((REPORT/'extraction_errors.json').read_text())
assert not errors, f'Falló la extracción de {len(errors)} audios: revisa extraction_errors.json.'
comparison=json.loads((REPORT/'model_comparison_status.json').read_text())
assert comparison['status']=='available' and comparison['n_pairs']==EXPECTED//2 and comparison['unmatched']==0, comparison
print('Pares comparados:',comparison['n_pairs'])
for figure in sorted(REPORT.glob('*.png')): display(Image(filename=str(figure)))
print((REPORT/'INTERPRETACION.md').read_text())

## Análisis para la revisión (comentarios 1, 3 y 5)

Las celdas 10–14 responden a los revisores con los mismos 60 WAV raw generados arriba:

- **Espacio normalizado común:** min–max sobre las 48 referencias + 60 síntesis juntas, con sensibilidad en z-score y unidades crudas.
- **Medidas de alineación:** $D$ (vs. perfil promedio de la categoría), $D^{nar}$ (vs. la grabación del mismo narrador usada como condicionante) y especificidad $S = \bar D^{otras} - D$.
- **Estadística con el video como unidad (n = 10):** Wilcoxon pareado entre modelos, Friedman entre narradores, Wilcoxon de $S>0$; efectos ($r_{rb}$, $W$), IC bootstrap 95 %, corrección de Holm y un modelo mixto `D ~ modelo × narrador + (1|video)`.
- **Corpus de referencia:** ICC(2,k) entre narradores para los 16 perfiles y prueba de variación entre categorías.
- **Descriptores extra (comentario 3):** variabilidad y rango de F0, variabilidad de sonoridad, picos de sonoridad/s.
- **Salidas:** tablas `.tex` ya llenas, figuras con dispersión y `RESULTADOS_MANUSCRITO.md` con frases que cambian según la significancia.

Copia `revision_analysis.py` **y la carpeta `fonts`** (DM Sans) dentro de **V01 Original** en Drive antes de ejecutar.

**Ambos modelos son obligatorios para la comparación:** la celda 6 debe haber generado los 60 WAV
(30 XTTS v2 + 30 Spanish-F5) con el mismo texto revisado. Las figuras y tablas se generan para los dos modelos
(`fig_deviation_by_model`, `fig_radar_videos_xtts`, `fig_radar_videos_spanish_f5`, `tab_alignment_model`, etc.).

In [ ]:
# @title 10. Instalar dependencias del análisis
assert (V01/'revision_analysis.py').exists(), 'Sube revision_analysis.py a la carpeta V01 Original.'
if not list((V01/'fonts').glob('DMSans*.ttf')): print('Aviso: no está V01/fonts con DM Sans; las figuras usarán otra fuente.')
_m=json.loads((OUT/'audio_manifest.json').read_text())
_eng={r['engine'] for r in _m}
print('Motores en el manifiesto:',{e:sum(r['engine']==e for r in _m) for e in _eng})
assert {'xtts','spanish_f5'}<=_eng, 'Falta un motor: ejecuta la celda 6 con ambos (opción 3) antes del análisis.'
run([UV,'pip','install','--python',PY,'opensmile','statsmodels','scipy','pandas','matplotlib','tabulate'])
# Seudónimos usados en el artículo. Verifica que el orden coincide con tu corpus.
PSEUDONYMS = {'Woman01':'Woman01','Man01':'Man01','Man02':'Man02'}
REV = OUT/'revision_estadistica'
print('Resultados de la revisión en:',REV)

In [ ]:
# @title 11. Extraer eGeMAPS, normalizar y ejecutar las pruebas
cmd=[PY,str(V01/'revision_analysis.py'),'--manifest',str(OUT/'audio_manifest.json'),
     '--references',str(REFS),'--output',str(REV),'--narrators',*NARRATORS,
     '--pseudonyms',*[f'{k}={v}' for k,v in PSEUDONYMS.items()]]
cached=REV/'features_raw_egemaps.csv'
if cached.exists(): cmd+=['--features-csv',str(cached)]   # evita re-extraer si ya existe
run(cmd)
import pandas as pd
cmap=pd.read_csv(REV/'reference_category_map.csv',index_col=0)
print('Categorías detectadas en Locutores (deben ser 16 filas con 1 en cada narrador):')
display(cmap)
assert len(cmap)==16 and (cmap==1).all().all(), 'Revisa nombres de archivos en Locutores antes de usar los resultados.'

In [ ]:
# @title 12. Revisar tablas de pruebas
pd.set_option('display.float_format','{:.3f}'.format)
for name in ['tests_model','tests_narrator','tests_narrator_posthoc','tests_specificity',
             'reference_icc','tests_reference_category_variation','descriptives_category']:
    print('\n==',name); display(pd.read_csv(REV/f'{name}.csv'))
print((REV/'mixed_model.txt').read_text())

In [ ]:
# @title 13. Figuras con dispersión (reemplazan o acompañan los radares)
for fig in ['fig_deviation_by_model','fig_deviation_by_narrator','fig_specificity','fig_reference_profiles']:
    print(fig); display(Image(filename=str(REV/'figures'/f'{fig}.png')))

In [ ]:
# @title 14. Texto y tablas LaTeX listos para el manuscrito
from IPython.display import Markdown
display(Markdown((REV/'RESULTADOS_MANUSCRITO.md').read_text(encoding='utf-8')))
for tex in sorted((REV/'tables').glob('*.tex')):
    print('\n%%%%%%', tex.name); print(tex.read_text())

In [ ]:
# @title 15. Descargar resultados, tablas y figuras
archive=shutil.make_archive('/content/AD_XTTS_SpanishF5_resultados','zip',OUT)
files.download(archive)
print('Resultados conservados en Drive:',OUT)

## Qué se obtiene

- WAV originales por video, motor y locutor, con registro de texto, referencia y versiones.
- Dispersión referencia/síntesis y tono/sonoridad; distribuciones con puntos individuales;
  correlaciones; perfiles emocionales y comparación emparejada XTTS–Spanish-F5.
- PNG, SVG y PDF; medidas eGeMAPS, tablas descriptivas y error con bootstrap por video.
- `text_preparation_audit.json`: texto original y texto preparado. No se modifican tus JSON fuente.

La comparación acústica no equivale a una evaluación perceptual de emoción o accesibilidad.
Los 10 videos son la unidad de muestreo; los tres locutores no son 30 videos independientes.
Los resúmenes largos pueden exceder el clip: esto no impide analizar el WAV raw; la mezcla
opcional avisa en lugar de cortar palabras. F5 transcribe localmente la referencia si
no existe un `.txt` homónimo; una transcripción exacta revisada mejora reproducibilidad.

La estructura y la preparación de las 10 entradas se validaron localmente. La generación
GPU real y la sesión de Colab deben ejecutarse para obtener resultados de ambos modelos.

El análisis de revisión (celdas 10–14) queda en `resultados_AD_existentes/revision_estadistica` y se incluye en el zip de la celda 9.
